In [1]:
import sys
import os

# --- Bước 1: Thêm thư mục gốc của dự án vào Python Path ---

# Lấy đường dẫn thư mục hiện tại của notebook (.../lesson-03/notebook)
current_dir = os.getcwd()

# Đi lùi 2 cấp để đến thư mục gốc của dự án (.../DS201-DL-PRACTICALLESSON)
project_root = os.path.abspath(os.path.join(current_dir, '..'))

# Thêm thư mục gốc vào sys.path nếu nó chưa có ở đó
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"Đã thêm vào sys.path: {project_root}")


Đã thêm vào sys.path: c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\lesson-03


In [2]:
from src.models.lstm import lstm
from src.utils import Vocab

from src.train import train

import torch
import torch.nn as nn
import pandas as pd
from functools import partial
from torch.utils.data import DataLoader

c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Session 00

### 0.1. Dataloader

In [3]:
from torch.utils.data import DataLoader, Dataset
import pandas as pd

In [4]:
VSVC_DATA_PATH = r'../dataset/uit_vsvc/'
VSVC_DATA_PATH

'../dataset/uit_vsvc/'

In [5]:
_train_vsvc = pd.read_json(
    VSVC_DATA_PATH + r'/UIT-VSFC-train.json'
)
_dev_vsvc = pd.read_json(
    VSVC_DATA_PATH + r'/UIT-VSFC-dev.json'
)
_test_vsvc = pd.read_json(
    VSVC_DATA_PATH + r'/UIT-VSFC-test.json'
)


In [6]:
vsvc_train_loader = DataLoader(
    dataset=_train_vsvc,
    batch_size=32,
    shuffle=True 
)
vsvc_dev_loader = DataLoader(
    dataset=_dev_vsvc,
    batch_size=32,
    shuffle=False 
)

## Session 01

### 1.1. Requirements

> Xây dựng mạng LSTM gồm 5 lớp với hidden size là 256 cho bài toán phân loại văn bản. Huấn luyện mô hình này trên bộ dữ liệu UIT-VSFC (Vietnamese Student Feedback Corpus) sử dụng Adam làm phương thức tối ưu tham số và đánh giá độ hiệu quả của mô hình sử dụng độ đo F1.

### 1.2. Configuration

#### 1.2.1. Vocabulary

In [7]:
vocab_path = r'..\dataset\uit_vsvc'
vocab = Vocab(
    vocab_path
)

In [8]:
PAD_IDX = vocab.w2i['<PAD>']

#### 1.2.2. Dataset

In [9]:
from src.utils import VsvcDataset, collate_fn

In [19]:
try:
    _train_vsvc = pd.read_json(vocab_path + r'\UIT-VSFC-train.json')
    _dev_vsvc = pd.read_json(vocab_path + r'\UIT-VSFC-dev.json')
    _test_vsvc = pd.read_json(vocab_path + r'\UIT-VSFC-test.json')
except FileNotFoundError:
    print(f"Lỗi: Không tìm thấy file train/dev tại '{vocab_path}'")

train_dataset = VsvcDataset(dataframe=_train_vsvc, vocab=vocab)
dev_dataset = VsvcDataset(dataframe=_dev_vsvc, vocab=vocab)
test_dataset = VsvcDataset(dataframe=_test_vsvc, vocab=vocab)

In [21]:
collate_fn_with_padding = partial(collate_fn, pad_idx=PAD_IDX)

# Tạo loader
vsvc_train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn_with_padding
)

vsvc_dev_loader = DataLoader(
    dataset=dev_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn_with_padding
)

vsvc_test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn_with_padding 
)

### 1.3. Training

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [13]:
lstm_instance = lstm(
    vocab_size=vocab.vocab_size,
    embedding_dim=256,
    hidden_size=256,
    output_size=vocab.n_labels
).to(device)

print(lstm_instance)

lstm(
  (embedding): Embedding(2879, 256, padding_idx=0)
  (lstm): LSTM(256, 256, batch_first=True)
  (dropout): Dropout(p=0.0, inplace=False)
  (fc): Linear(in_features=256, out_features=4, bias=True)
)


In [24]:
optimizer = torch.optim.Adam(lstm_instance.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

NUM_EPOCHS = 50
N_LABELS = vocab.n_labels 
MODEL_PATH   = r"..\checkpoints\lstm\lstm_best_model.pth"
HISTORY_PATH = r"..\checkpoints\lstm\training_history.json"

In [25]:
best_model, history_data = train(
    model=lstm_instance,
    train_loader=vsvc_train_loader,
    val_loader=vsvc_dev_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    num_epochs=NUM_EPOCHS,
    n_labels=N_LABELS,
    model_save_path=MODEL_PATH,
    history_save_path=HISTORY_PATH,
    patience=10
)

--- Bắt đầu training ---
Lưu model tốt nhất tại: ..\checkpoints\lstm\lstm_best_model.pth
Lưu lịch sử training tại: ..\checkpoints\lstm\training_history.json


Epoch 1/50 [Validation]: 100%|██████████| 50/50 [00:00<00:00, 195.73it/s]


Epoch 1: Train Loss: 0.2812 | Val Loss: 0.3733 | Val F1: 0.8680
🎉 New best F1: 0.8680. Model saved to ..\checkpoints\lstm\lstm_best_model.pth


Epoch 2/50 [Validation]: 100%|██████████| 50/50 [00:00<00:00, 219.81it/s]


Epoch 2: Train Loss: 0.2101 | Val Loss: 0.3881 | Val F1: 0.8705
🎉 New best F1: 0.8705. Model saved to ..\checkpoints\lstm\lstm_best_model.pth


...

Epoch 12: Train Loss: 0.0262 | Val Loss: 0.7380 | Val F1: 0.8636
No improvement. Patience: 8/10


Epoch 13/50 [Validation]: 100%|██████████| 50/50 [00:00<00:00, 212.62it/s]


Epoch 13: Train Loss: 0.0196 | Val Loss: 0.7843 | Val F1: 0.8648
No improvement. Patience: 9/10


Epoch 14/50 [Validation]: 100%|██████████| 50/50 [00:00<00:00, 226.41it/s]

Epoch 14: Train Loss: 0.0246 | Val Loss: 0.7991 | Val F1: 0.8560
No improvement. Patience: 10/10
Early stopping triggered after 14 epochs.

--- Training finished ---
Best Validation F1-score: 0.8711
Training history successfully saved to ..\checkpoints\lstm\training_history.json


### 1.4. Evaluation

In [26]:
from src.evaluate import evaluate

In [27]:
label_names = [vocab.i2l[i] for i in range(vocab.n_labels)]
print(f"Các nhãn (theo thứ tự): {label_names}")

Các nhãn (theo thứ tự): ['training_program', 'lecturer', 'others', 'facility']


In [28]:
test_results = evaluate(
    model=best_model,
    test_loader=vsvc_test_loader,
    criterion=criterion,
    device=device,
    n_labels=N_LABELS,
    label_names=label_names
)

--- Bắt đầu đánh giá trên tập Test ---


Evaluating: 100%|██████████| 99/99 [00:00<00:00, 181.36it/s]


--- 🏁 Kết quả Đánh giá trên tập Test ---
Thời gian đánh giá: 0.55 giây
Test Loss: 	0.4517
Test Accuracy: 	86.86%
Test F1-Score (Macro): 	0.7472

📊 Báo cáo chi tiết (Classification Report):
                  precision    recall  f1-score   support

training_program       0.72      0.73      0.73       572
        lecturer       0.94      0.93      0.93      2290
          others       0.45      0.47      0.46       159
        facility       0.88      0.87      0.87       145

        accuracy                           0.87      3166
       macro avg       0.75      0.75      0.75      3166
    weighted avg       0.87      0.87      0.87      3166

